# Data Collection and Preprocessing

Video 91 starter. The planned workflow will use Shakespeare's Hamlet for text collection, tokenization, sequence generation, padding, and preprocessing. Implementation will be added one course video at a time.

In [2]:
import sys
print(sys.executable)

d:\AI_Projects_Practice\LSTM\.venv\Scripts\python.exe


In [9]:
import nltk

nltk.download('gutenberg')

from nltk.corpus import gutenberg

data = gutenberg.raw('shakespeare-hamlet.txt')

with open('hamlet.txt', 'w', encoding='utf-8') as file:
    file.write(data)

print("Hamlet dataset saved successfully!")
print(data[:500])

Hamlet dataset saved successfully!
[The Tragedie of Hamlet by William Shakespeare 1599]


Actus Primus. Scoena Prima.

Enter Barnardo and Francisco two Centinels.

  Barnardo. Who's there?
  Fran. Nay answer me: Stand & vnfold
your selfe

   Bar. Long liue the King

   Fran. Barnardo?
  Bar. He

   Fran. You come most carefully vpon your houre

   Bar. 'Tis now strook twelue, get thee to bed Francisco

   Fran. For this releefe much thankes: 'Tis bitter cold,
And I am sicke at heart

   Barn. Haue you had quiet Guard?
  Fran. Not


[nltk_data] Downloading package gutenberg to C:\Users\chandra
[nltk_data]     sekhar\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [10]:
with open('hamlet.txt', 'r', encoding='utf-8') as file:
    text = file.read().lower()

print(text[:500])

[the tragedie of hamlet by william shakespeare 1599]


actus primus. scoena prima.

enter barnardo and francisco two centinels.

  barnardo. who's there?
  fran. nay answer me: stand & vnfold
your selfe

   bar. long liue the king

   fran. barnardo?
  bar. he

   fran. you come most carefully vpon your houre

   bar. 'tis now strook twelue, get thee to bed francisco

   fran. for this releefe much thankes: 'tis bitter cold,
and i am sicke at heart

   barn. haue you had quiet guard?
  fran. not


In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

print("Total words:", total_words)
print("Sample word indexes:", list(tokenizer.word_index.items())[:10])

Total words: 4818
Sample word indexes: [('the', 1), ('and', 2), ('to', 3), ('of', 4), ('i', 5), ('you', 6), ('a', 7), ('my', 8), ('it', 9), ('in', 10)]


In [12]:
input_sequences = []

for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print("First 10 sequences:")
print(input_sequences[:10])

First 10 sequences:
[[1, 687], [1, 687, 4], [1, 687, 4, 45], [1, 687, 4, 45, 41], [1, 687, 4, 45, 41, 1886], [1, 687, 4, 45, 41, 1886, 1887], [1, 687, 4, 45, 41, 1886, 1887, 1888], [1180, 1889], [1180, 1889, 1890], [1180, 1889, 1890, 1891]]


In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

max_sequence_len = max(len(seq) for seq in input_sequences)

input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')
)

print("Maximum sequence length:", max_sequence_len)
print("Padded sequences shape:", input_sequences.shape)
print("First 5 padded sequences:")
print(input_sequences[:5])

Maximum sequence length: 14
Padded sequences shape: (25732, 14)
First 5 padded sequences:
[[   0    0    0    0    0    0    0    0    0    0    0    0    1  687]
 [   0    0    0    0    0    0    0    0    0    0    0    1  687    4]
 [   0    0    0    0    0    0    0    0    0    0    1  687    4   45]
 [   0    0    0    0    0    0    0    0    0    1  687    4   45   41]
 [   0    0    0    0    0    0    0    0    1  687    4   45   41 1886]]


In [17]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("First X sample:", X[0])
print("First y sample:", y[0])

X shape: (25732, 13)
y shape: (25732,)
First X sample: [0 0 0 0 0 0 0 0 0 0 0 0 1]
First y sample: 687


In [18]:
# Keep target word IDs as integers to avoid a large one-hot matrix.
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target dtype:", y.dtype)
print("Use loss='sparse_categorical_crossentropy' when compiling the model.")

X shape: (25732, 13)
y shape: (25732,)
Target dtype: int32
Use loss='sparse_categorical_crossentropy' when compiling the model.


In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (20585, 13)
X_test: (5147, 13)
y_train: (20585,)
y_test: (5147,)


In [22]:
#Why Softmax? The task is multiclass classification: the model chooses among many possible next words.
import pickle

with open("preprocessed_data.pkl", "wb") as f:
    pickle.dump({
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "total_words": total_words,
        "max_sequence_len": max_sequence_len,
        "tokenizer": tokenizer
    }, f)

print("Preprocessed data saved successfully!")

Preprocessed data saved successfully!


In [23]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Check the original target shape
print("Original y shape:", y.shape)

# One-hot encode only if y contains integer labels
if y.ndim == 1:
    y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# Split again using the encoded labels
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Original y shape: (25732,)
X_train: (20585, 13)
X_test: (5147, 13)
y_train: (20585, 4818)
y_test: (5147, 4818)


In [24]:
import pickle

with open("preprocessed_data.pkl", "wb") as f:
    pickle.dump({
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "total_words": total_words,
        "max_sequence_len": max_sequence_len,
        "tokenizer": tokenizer
    }, f)

print("Preprocessed data saved successfully!")

Preprocessed data saved successfully!
